# Leishmania mexicana AmpB Resistance — Snakemake Pipeline (Colab run)

This notebook is a Snakemake reimplementation of the genomic variant-calling workflow from my [Integrative Multi-Omics Analysis of Amphotericin B Resistance in *Leishmania mexicana*](https://github.com/Claire-bioinformatics/Integrative-Multi-Omics-Analysis-of-Amphotericin-B-Resistance-in-Leishmania-mexicana) project, end-to-end against **public data**, since the original university HPC data is no longer accessible.

**Data source:** Mwenechanya et al. (2017), *Sterol 14α-demethylase mutation leads to amphotericin B resistance in Leishmania mexicana*, PLOS NTD 11(6):e0005649. Raw WGS reads: ENA project [PRJEB10872](https://www.ebi.ac.uk/ena/browser/view/PRJEB10872), runs `ERR1517168` (WT) and `ERR1517169` (AmpB-resistant), strain M379, Illumina GAIIx.

**Reference genome:** *L. mexicana* MHOM/GT/2001/U1103, NCBI assembly `GCA_000234665.4`.

## 1. Mount Google Drive
Only used at the very end to save final results

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Install conda + the pipeline's tool environment
`condacolab` swaps Colab's Python environment for a conda one, which requires a runtime restart. The next cell **will crash/restart the kernel on purpose** — ignore the "Session crashed" message, that's condacolab doing its job. Just continue with the cell after it once the restart finishes.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


## 3. Install bioinformatics tools (post-restart)
Mirrors the repo's `environment.yml`, plus the NCBI `datasets` CLI for pulling the reference genome. This takes a few minutes.

In [ ]:
import condacolab
condacolab.check()  # confirms the restart from Section 2 actually happened

!mamba install -y -c bioconda -c conda-forge \
    snakemake-minimal=8.* fastqc=0.12.* trim-galore=0.6.* bowtie2=2.5.* \
    samtools=1.20.* bamaddrg freebayes=1.3.* vcflib=1.0.* bcftools=1.20.* \
    snpeff=5.2 snpsift=5.2 ncbi-datasets-cli

✨🍰✨ Everything looks OK!

Looking for: ['snakemake-minimal=8', 'fastqc=0.12', 'trim-galore=0.6', 'bowtie2=2.5', 'samtools=1.20', 'bamaddrg', 'freebayes=1.3', 'vcflib=1.0', 'bcftools=1.20', 'snpeff=5.2', 'snpsift=5.2', 'ncbi-datasets-cli']

[+] 0.0s
bioconda/linux-64                                             No change
[+] 0.1s
bioconda/noarch       ⣾  
conda-forge/linux-64  ⣾  
conda-forge/noarch    ⣾  [+] 0.2s
bioconda/noarch       11%
conda-forge/linux-64   3%
conda-forge/noarch     4%[+] 0.3s
bioconda/noarch       30%
conda-forge/linux-64   5%
conda-forge/noarch     8%[+] 0.4s
bioconda/noarch       52%
conda-forge/linux-64   7%
conda-forge/noarch    13%[+] 0.5s
bioconda/noarch       65%
conda-forge/linux-64   8%
conda-forge/noarch    15%[+] 0.6s
bioconda/noarch       91%
conda-forge/linux-64  11%
conda-forge/noarch    21%[+] 0.7s
bioconda/noarch      100%
conda-forge/linux-64  12%
conda-forge/noarch    23%bioconda/noarch                                   
[+] 0.8s
conda-forge/linux

Troubleshooting for cell 3.<br>

If cell 3 got stuck, run the next 2 cells of code, then re-run cell 3

In [ ]:
!cat /usr/local/conda-meta/pinned

cat: /usr/local/conda-meta/pinned: No such file or directory


In [ ]:
!rm /usr/local/conda-meta/pinned

rm: cannot remove '/usr/local/conda-meta/pinned': No such file or directory


In [ ]:
!snakemake --version && bowtie2 --version | head -1 && samtools --version | head -1 && snpEff -version

8.30.0
/usr/local/bin/bowtie2-align-s version 2.5.4
samtools 1.20
SnpEff	5.2	2023-09-29


## 4. Clone the pipeline repo

In [ ]:
!git clone https://github.com/Claire-bioinformatics/snakemake-leishmania-ampb-resistance.git /content/pipeline
%cd /content/pipeline

fatal: destination path '/content/pipeline' already exists and is not an empty directory.
/content/pipeline


## 5. Download raw reads from ENA

In [ ]:
!mkdir -p data
!wget -q --show-progress -O data/WT_1.fastq.gz   ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR151/008/ERR1517168/ERR1517168_1.fastq.gz
!wget -q --show-progress -O data/WT_2.fastq.gz   ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR151/008/ERR1517168/ERR1517168_2.fastq.gz
!wget -q --show-progress -O data/AmpB_1.fastq.gz ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR151/009/ERR1517169/ERR1517169_1.fastq.gz
!wget -q --show-progress -O data/AmpB_2.fastq.gz ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR151/009/ERR1517169/ERR1517169_2.fastq.gz
!ls -lh data/

ERR1517168_1.fastq. 100%[===================>]   2.02G  29.7MB/s    in 74s     
ERR1517168_2.fastq. 100%[===================>]   2.10G  40.1MB/s    in 92s     
ERR1517169_1.fastq. 100%[===================>]   1.71G  18.3MB/s    in 50s     
ERR1517169_2.fastq. 100%[===================>]   1.78G  40.5MB/s    in 48s     
total 7.7G
-rw-r--r-- 1 root root 1.8G Jul 12 11:31 AmpB_1.fastq.gz
-rw-r--r-- 1 root root 1.8G Jul 12 11:31 AmpB_2.fastq.gz
-rw-r--r-- 1 root root 2.1G Jul 12 11:28 WT_1.fastq.gz
-rw-r--r-- 1 root root 2.2G Jul 12 11:30 WT_2.fastq.gz


## 6. Check the quality-score encoding
The Snakefile's `trim_galore`/`bowtie2` rules assume `--phred64` (true for the original GAIIx coursework files). ENA sometimes re-encodes submissions to standard Phred33, so check before trusting that flag rather than assuming it still holds.

In [ ]:
import gzip
with gzip.open('data/WT_1.fastq.gz', 'rt') as f:
    for i, line in enumerate(f):
        if i == 3:  # 4th line of the first FASTQ record = quality string
            quals = line.strip()
            break
print("Quality string sample:", quals)
print("ASCII range:", min(ord(c) for c in quals), "-", max(ord(c) for c in quals))
print("-> Phred64 if max ASCII > ~74; Phred33 if max ASCII <= ~74.")

Quality string sample: #######################################################################################################################################################
ASCII range: 35 - 35
-> Phred64 if max ASCII > ~74; Phred33 if max ASCII <= ~74.


In [ ]:
import gzip
with gzip.open('data/WT_1.fastq.gz', 'rt') as f:
    for i, line in enumerate(f):
        if i >= 40:  # stop after checking 10 records (4 lines each)
            break
        if i % 4 == 1:
            print("seq: ", line.strip()[:60])
        if i % 4 == 3:
            quals = line.strip()
            print("qual:", quals[:60], " | unique chars:", set(quals))

seq:  NGAGGAGAAGCAACAGCAGAACATCCGCGATCATCAAGATCATCATCTGATGGAGAAACG
qual: ############################################################  | unique chars: {'#'}
seq:  NTGCGCTTGTTGTGGGGATTTTGTGTGATTTGTTATTGACGTTTTAGATTTAAATGTTTT
qual: #++++-3/.-@@@@@5005785800.2222:::22:::.:<::<::::::@@@@@@@@@2  | unique chars: {':', '#', '+', '3', '0', '8', '1', '9', '.', '7', '2', '5', '<', '@', '-', '/'}
seq:  NTCCTGTGTGCGGGCGGTGCCGCGAAAGTAGGGTTGCAGCCAGCACACCAAGCTCGGGAT
qual: #2///2.--.883383../5:::::<:<:<::::::8:::@@@@@@@@@@##########  | unique chars: {':', '#', '3', '@', '.', '2', '5', '<', '8', '-', '/'}
seq:  NCCGTTCTGGTTCCCGGCGTAGTCAGACGTGGAGAAGATGGTGACGACACGGGCATTGTC
qual: #+-//25552@@@@@<:<:<<<8<<@@@@@@@@:@@@@@@::::::<<<<:8-:::::::  | unique chars: {':', '#', '+', '8', '2', '5', '<', '@', '-', '/'}
seq:  NTGTAAAGTAAATTCAAGTGATACAATAATCATAACGCACGCATATCATCTATTAAATAC
qual: #,))+-,+--78777@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@  | unique chars: {'#', ',', '+', '@', '7', '2', '8', '-', ')'}
seq: 

## 7. Download the reference genome + annotation
*L. mexicana* MHOM/GT/2001/U1103, NCBI assembly GCA_000234665.4, via the `datasets` CLI. Placed at `data/reference/Lmexicana.fasta`, matching `config.yaml`'s default path so `config.yaml` doesn't need editing for this part.

In [ ]:
!mkdir -p data/reference
!datasets download genome accession GCA_000234665.4 --include genome,gff3 --filename /content/genome.zip
!unzip -q -o /content/genome.zip -d /content/genome_dl

import glob, shutil
fasta_src = glob.glob('/content/genome_dl/ncbi_dataset/data/GCA_000234665.4/*.fna')[0]
gff_src   = glob.glob('/content/genome_dl/ncbi_dataset/data/GCA_000234665.4/*.gff')[0]
shutil.copy(fasta_src, 'data/reference/Lmexicana.fasta')
shutil.copy(gff_src,   'data/reference/Lmexicana.gff')
!ls -lh data/reference/

Downloading: /content/genome.zip    2.33kB 15.7MB/s
Downloading: /content/genome.zip    2.33kB 15.7MB/s
Downloading: /content/genome.zip    2.33kB 15.7MB/s
Downloading: /content/genome.zip    2.33kB 15.7MB/s
Downloading: /content/genome.zip    3.9kB 80.2kB/s
Downloading: /content/genome.zip    3.9kB 80.2kB/s
Downloading: /content/genome.zip    18.1kB 249kB/s
Downloading: /content/genome.zip    32.8kB 446kB/s
Downloading: /content/genome.zip    47.1kB 504kB/s
Downloading: /content/genome.zip    65.5kB 657kB/s
Downloading: /content/genome.zip    65.5kB 657kB/s
Downloading: /content/genome.zip    197kB 1.56MB/s
Downloading: /content/genome.zip    197kB 1.56MB/s
Downloading: /content/genome.zip    197kB 1.56MB/s
Downloading: /content/genome.zip    426kB 2.76MB/s
Downloading: /content/genome.zip    426kB 2.76MB/s
Downloading: /content/genome.zip    524kB 2.92MB/s
Downloading: /content/genome.zip    852kB 4.6MB/s
Downloading: /content/genome.zip    918kB 4.64MB/s
Downloading: /content/genome

## 8. Build the custom SnpEff database (`Lmex`)
`snpeff_annotate` in the Snakefile calls `snpEff ... Lmex {input}` but `Lmex` isn't one of SnpEff's pre-built genomes, so it has to be built once from the reference + GFF before the pipeline can annotate anything. The Snakefile's `-noCheckCds -noCheckProtein` flags mean we don't need separate CDS/protein FASTAs for this build.

In [ ]:
import os
os.makedirs('config/data/Lmex', exist_ok=True)
shutil.copy('data/reference/Lmexicana.fasta', 'config/data/Lmex/sequences.fa')
shutil.copy('data/reference/Lmexicana.gff',   'config/data/Lmex/genes.gff')

with open('config/snpEff.config', 'w') as f:
    f.write('data.dir = /content/pipeline/config/data/\n')
    f.write('Lmex.genome : Leishmania_mexicana\n')

!snpEff build -gff3 -v Lmex -c config/snpEff.config -noCheckCds -noCheckProtein

00:00:00 SnpEff version SnpEff 5.2 (build 2023-09-29 06:17), by Pablo Cingolani
00:00:00 Command: 'build'
00:00:00 Building database for 'Lmex'
00:00:00 Reading configuration file 'config/snpEff.config'. Genome: 'Lmex'
00:00:00 Reading config file: /content/pipeline/config/snpEff.config
00:00:00 done
00:00:00 Reading GFF3 data file  : '/content/pipeline/config/data/Lmex/genes.gff'
00:00:00 Reading file '/content/pipeline/config/data/Lmex/genes.gff'
WARNING_TRANSCRIPT_NOT_FOUND: Exon's parent 'gene-LMXM_01_0010' is a Gene instead of a transcript. Created transcript 'TRANSCRIPT_gene-LMXM_01_0010' for FR799554.1	EMBL	CDS	3967	4971	-
	dbxref : InterPro:IPR021333,UniProtKB/TrEMBL:E9AJ87,NCBI_GP:CBZ22984.1
	gbkey : CDS
	id : cds-CBZ22984.1
	locus_tag : LMXM_01_0010
	name : CBZ22984.1
	parent : gene-LMXM_01_0010
	product : hypothetical protein%2C unknown function
	protein_id : CBZ22984.1
	source : EMBL
	type : CDS
. File '/content/pipeline/config/data/Lmex/genes.gff' line 10	'FR799554.1	EMBL	

## 9. Dry run

In [ ]:
!snakemake -n

host: f10a3993de0f
Building DAG of jobs...
Job stats:
job                      count
---------------------  -------
align_sort                   2
all                          1
bowtie2_build                1
call_snps                    1
extract_nonsynonymous        1
fastqc                       2
filter_snps                  1
index_bam                    2
snpeff_annotate              1
split_sample                 2
trim_galore                  2
unique_ampb_snps             1
total                       17


[Sun Jul 12 11:57:05 2026]
rule trim_galore:
    input: data/WT_1.fastq.gz, data/WT_2.fastq.gz
    output: results/trimmed/WT_1_val_1.fq.gz, results/trimmed/WT_2_val_2.fq.gz
    jobid: 8
    reason: Missing output files: results/trimmed/WT_2_val_2.fq.gz, results/trimmed/WT_1_val_1.fq.gz
    wildcards: sample=WT
    resources: tmpdir=<TBD>


[Sun Jul 12 11:57:05 2026]
rule trim_galore:
    input: data/AmpB_1.fastq.gz, data/AmpB_2.fastq.gz
    output: results/trimmed/AmpB_1_va

## 10. Run the pipeline
`--cores 2` matches Colab's free-tier vCPU count. This is the long step — QC, trimming, alignment, and variant calling on ~10 GB of reads. Keep the tab open/active so Colab doesn't idle-disconnect mid-run.

In [ ]:
!snakemake --cores 2 --rerun-incomplete

Assuming unrestricted shared filesystem usage.
host: f10a3993de0f
Building DAG of jobs...
Using shell: /usr/bin/bash
Provided cores: 2
Rules claiming more threads will be scaled down.
Job stats:
job                      count
---------------------  -------
all                          1
extract_nonsynonymous        1
total                        2

Select jobs to execute...
Execute 1 jobs...

[Mon Jul 13 02:27:36 2026]
localrule extract_nonsynonymous:
    input: results/snpeff/AmpB_unique.ann.vcf
    output: results/snpeff/AmpB_unique_nonsyn.tsv
    jobid: 1
    reason: Missing output files: results/snpeff/AmpB_unique_nonsyn.tsv
    resources: tmpdir=/tmp

[Mon Jul 13 02:27:37 2026]
Finished job 1.
1 of 2 steps (50%) done
Select jobs to execute...
Execute 1 jobs...

[Mon Jul 13 02:27:37 2026]
localrule all:
    input: results/snpeff/AmpB_unique_nonsyn.tsv, results/fastqc/WT_1_fastqc.html, results/fastqc/WT_2_fastqc.html, results/fastqc/AmpB_1_fastqc.html, results/fastqc/AmpB_2_fastqc.h

In [ ]:
!rm -f results/aligned/*.tmp.*.bam
!ls results/aligned/

AmpB.bowtie2.log  WT.bowtie2.log


In [ ]:
!find / -iname "vcfEffOnePerLine.pl" 2>/dev/null

/usr/local/share/snpsift-5.2-0/scripts/vcfEffOnePerLine.pl
/usr/local/share/snpeff-5.2-3/scripts/vcfEffOnePerLine.pl
/usr/local/pkgs/snpeff-5.2-hdfd78af_3/share/snpeff-5.2-3/scripts/vcfEffOnePerLine.pl
/usr/local/pkgs/snpsift-5.2-hdfd78af_0/share/snpsift-5.2-0/scripts/vcfEffOnePerLine.pl


In [ ]:
!chmod +x /usr/local/share/snpsift-5.2-0/scripts/vcfEffOnePerLine.pl
!ln -s /usr/local/share/snpsift-5.2-0/scripts/vcfEffOnePerLine.pl /usr/local/bin/vcfEffOnePerLine.pl
!which vcfEffOnePerLine.pl

/usr/local/bin/vcfEffOnePerLine.pl


##11. Check for CYP51


In [ ]:
import pandas as pd
cols = ['CHROM','POS','REF','ALT','IMPACT','EFFECT','GENE','HGVS_C','HGVS_P','GT']
df = pd.read_csv('results/snpeff/AmpB_unique_nonsyn.tsv', sep='\t', header=None, names=cols)
print(f"{len(df)} non-synonymous / high-impact SNPs unique to the AmpB-resistant strain")
df[df['GENE'].str.contains('11.1100', na=False)]  # CYP51's TriTrypDB gene ID from your report
df.head(20)

52 non-synonymous / high-impact SNPs unique to the AmpB-resistant strain


,CHROM,POS,REF,ALT,IMPACT,EFFECT,GENE,HGVS_C,HGVS_P,GT
0,FR799555.1,122038,C,G,MODERATE,missense_variant,LMXM_02_0310,c.1535G>C,p.Arg512Pro,0/1
1,FR799555.1,242695,G,T,MODERATE,missense_variant,LMXM_02_0580,c.1779G>T,p.Glu593Asp,0/1
2,FR799556.1,67844,C,G,MODERATE,missense_variant,LMXM_03_0270,c.726C>G,p.Asp242Glu,1/1
3,FR799556.1,70050,G,C,MODERATE,missense_variant,LMXM_03_0270,c.2932G>C,p.Val978Leu,0/1
4,FR799556.1,97488,C,T,MODERATE,missense_variant,LMXM_03_0350,c.1381C>T,p.Pro461Ser,0/1
5,FR799556.1,106289,C,T,MODERATE,missense_variant,LMXM_03_0360,c.3029C>T,p.Ala1010Val,0/1
6,FR799556.1,165044,G,A,MODERATE,missense_variant,LMXM_03_0490,c.298G>A,p.Gly100Ser,0/1
7,FR799556.1,179643,T,A,MODERATE,missense_variant,LMXM_03_0510,c.6395T>A,p.Val2132Glu,0/1
8,FR799556.1,191350,G,A,MODERATE,missense_variant,LMXM_03_0530,c.5320G>A,p.Asp1774Asn,0/1
9,FR799557.1,392587,TGTGTGAC,TGTGTGTC,MODERATE,missense_variant,LMXM_04_1140,c.29T>A,p.Val10Asp,./1


##12. Save to Drive

In [ ]:
import os
dest = '/content/drive/MyDrive/Leishmania_AmpB_results'
os.makedirs(dest, exist_ok=True)
!cp -r results/snpeff {dest}/
!cp -r results/fastqc {dest}/
!cp results/aligned/*.bowtie2.log {dest}/ 2>/dev/null || true
!du -sh {dest}
print(f"Saved to {dest}")

4.5M	/content/drive/MyDrive/Leishmania_AmpB_results
Saved to /content/drive/MyDrive/Leishmania_AmpB_results
